In [16]:
"""
Run this in terminal to build docker and run fastapi model service

docker build -t mortgage-default-api .
docker run -p 8000:8000 mortgage-default-api
"""

'\nRun this in terminal to build docker and run fastapi model service\ndocker build -t mortgage-default-api .\ndocker run -p 8000:8000 mortgage-default-api'

In [65]:
import json
import requests
import pandas as pd

BASE_URL = "http://127.0.0.1:8000"

def pretty(response):
    print("Status:", response.status_code)
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text)

In [66]:
response = requests.get(f"{BASE_URL}/health")
pretty(response)

assert response.status_code == 200

Status: 200
{
  "model_ready": true,
  "model_uri": "data/06_models/mlflow_model",
  "metadata_path": "data/08_reporting/model_training_metadata.json",
  "feature_transformers_path": "data/04_feature/feature_transformers.pkl",
  "error": null,
  "status": "ok"
}


In [67]:
response = requests.get(f"{BASE_URL}/metadata")
pretty(response)

assert response.status_code == 200

Status: 200
{
  "model_uri": "data/06_models/mlflow_model",
  "selected_model_family": "random_forest",
  "threshold": 0.5,
  "target_column": "default",
  "id_column": "loan_sequence_number",
  "n_features": 94
}


In [68]:
loan_1 = {
    "loan_sequence_number": "TEST_001",
    "credit_score": 720,
    "first_payment_date": 200503,
    "first_time_homebuyer_flag": "N",
    "maturity_date": 203502,
    "mi_percentage": 0,
    "number_of_units": 1,
    "occupancy_status": "P",
    "original_cltv": 80,
    "original_dti": 35,
    "original_upb": 200000,
    "original_ltv": 80,
    "original_interest_rate": 5.5,
    "channel": "R",
    "prepayment_penalty_flag": "N",
    "amortization_type": "FRM",
    "interest_only_indicator": "N",
    "property_state": "CA",
    "property_type": "SF",
    "postal_code": "90000",
    "loan_purpose": "P",
    "original_loan_term": 360,
    "number_of_borrowers": 2,
    "seller_name": "OTHER",
    "servicer_name": "OTHER",
    "super_conforming_flag": "N",
}

payload_one = {
    "features": loan_1
}
loan_1

{'loan_sequence_number': 'TEST_001',
 'credit_score': 720,
 'first_payment_date': 200503,
 'first_time_homebuyer_flag': 'N',
 'maturity_date': 203502,
 'mi_percentage': 0,
 'number_of_units': 1,
 'occupancy_status': 'P',
 'original_cltv': 80,
 'original_dti': 35,
 'original_upb': 200000,
 'original_ltv': 80,
 'original_interest_rate': 5.5,
 'channel': 'R',
 'prepayment_penalty_flag': 'N',
 'amortization_type': 'FRM',
 'interest_only_indicator': 'N',
 'property_state': 'CA',
 'property_type': 'SF',
 'postal_code': '90000',
 'loan_purpose': 'P',
 'original_loan_term': 360,
 'number_of_borrowers': 2,
 'seller_name': 'OTHER',
 'servicer_name': 'OTHER',
 'super_conforming_flag': 'N'}

In [69]:
payload = {
    "features": loan_1
}

response = requests.post(f"{BASE_URL}/predict-one", json=payload)
pretty(response)

assert response.status_code == 200

Status: 200
{
  "loan_sequence_number": "TEST_001",
  "prediction": 0,
  "probability_default": 0.3926324924934179,
  "threshold": 0.5,
  "model_uri": "data/06_models/mlflow_model"
}


In [70]:
payload = {
    "features": loan_1,
    "threshold": 0.3
}

response = requests.post(f"{BASE_URL}/predict-one", json=payload)
pretty(response)

assert response.status_code == 200

Status: 200
{
  "loan_sequence_number": "TEST_001",
  "prediction": 1,
  "probability_default": 0.3926324924934179,
  "threshold": 0.3,
  "model_uri": "data/06_models/mlflow_model"
}


In [71]:
loan_2 = {
    "loan_sequence_number": "TEST_002",
    "credit_score": 620,
    "first_payment_date": 200503,
    "first_time_homebuyer_flag": "N",
    "maturity_date": 203502,
    "mi_percentage": 0,
    "number_of_units": 1,
    "occupancy_status": "P",
    "original_cltv": 95,
    "original_dti": 48,
    "original_upb": 260000,
    "original_ltv": 95,
    "original_interest_rate": 7.0,
    "channel": "R",
    "prepayment_penalty_flag": "N",
    "amortization_type": "FRM",
    "interest_only_indicator": "N",
    "property_state": "FL",
    "property_type": "SF",
    "postal_code": "33000",
    "loan_purpose": "C",
    "original_loan_term": 360,
    "number_of_borrowers": 1,
    "seller_name": "OTHER",
    "servicer_name": "OTHER",
    "super_conforming_flag": "N",
}

payload_batch = {
    "rows": [loan_1, loan_2]
}

loan_3 = loan_1.copy()
loan_3.update({
    "loan_sequence_number": "TEST_003",
    "credit_score": 780,
    "original_dti": 25,
    "original_ltv": 65,
    "original_cltv": 65,
    "original_interest_rate": 4.9,
    "property_state": "TX",
    "loan_purpose": "N",
})

batch_payload = {
    "rows": [loan_1, loan_2, loan_3]
}

response = requests.post(f"{BASE_URL}/predict", json=batch_payload)
pretty(response)

assert response.status_code == 200

Status: 200
{
  "n_rows": 3,
  "predictions": [
    {
      "loan_sequence_number": "TEST_001",
      "prediction": 0,
      "probability_default": 0.3926324924934179,
      "threshold": 0.5,
      "model_uri": "data/06_models/mlflow_model"
    },
    {
      "loan_sequence_number": "TEST_002",
      "prediction": 1,
      "probability_default": 0.556764528711197,
      "threshold": 0.5,
      "model_uri": "data/06_models/mlflow_model"
    },
    {
      "loan_sequence_number": "TEST_003",
      "prediction": 0,
      "probability_default": 0.12267098762619925,
      "threshold": 0.5,
      "model_uri": "data/06_models/mlflow_model"
    }
  ]
}


In [72]:
batch_result = response.json()

predictions_df = pd.DataFrame(batch_result["predictions"])
predictions_df

,loan_sequence_number,prediction,probability_default,threshold,model_uri
0,TEST_001,0,0.392632,0.5,data/06_models/mlflow_model
1,TEST_002,1,0.556765,0.5,data/06_models/mlflow_model
2,TEST_003,0,0.122671,0.5,data/06_models/mlflow_model


In [73]:
response = requests.post(f"{BASE_URL}/reload-model")
pretty(response)

assert response.status_code == 200

Status: 200
{
  "model_ready": true,
  "model_uri": "data/06_models/mlflow_model",
  "metadata_path": "data/08_reporting/model_training_metadata.json",
  "feature_transformers_path": "data/04_feature/feature_transformers.pkl",
  "error": null,
  "status": "ok"
}


In [74]:
checks = {}

checks["health"] = requests.get(f"{BASE_URL}/health").status_code
checks["metadata"] = requests.get(f"{BASE_URL}/metadata").status_code
checks["predict_one"] = requests.post(
    f"{BASE_URL}/predict-one",
    json={"features": loan_1},
).status_code
checks["predict_batch"] = requests.post(
    f"{BASE_URL}/predict",
    json={"rows": [loan_1, loan_2]},
).status_code

checks

{'health': 200, 'metadata': 200, 'predict_one': 200, 'predict_batch': 200}

In [75]:
import json
import requests

BASE_URL = "http://127.0.0.1:8000"

r = requests.post(
    f"{BASE_URL}/predict-one",
    json={"features": loan_1},
)

print("Status:", r.status_code)
print(json.dumps(r.json(), indent=2))

Status: 200
{
  "loan_sequence_number": "TEST_001",
  "prediction": 0,
  "probability_default": 0.3926324924934179,
  "threshold": 0.5,
  "model_uri": "data/06_models/mlflow_model"
}


In [76]:
r = requests.post(
    f"{BASE_URL}/predict",
    json={"rows": [loan_1, loan_2]},
)

print("Status:", r.status_code)
print(json.dumps(r.json(), indent=2))

Status: 200
{
  "n_rows": 2,
  "predictions": [
    {
      "loan_sequence_number": "TEST_001",
      "prediction": 0,
      "probability_default": 0.3926324924934179,
      "threshold": 0.5,
      "model_uri": "data/06_models/mlflow_model"
    },
    {
      "loan_sequence_number": "TEST_002",
      "prediction": 1,
      "probability_default": 0.556764528711197,
      "threshold": 0.5,
      "model_uri": "data/06_models/mlflow_model"
    }
  ]
}
